# Neutral Sentence Generation

## Load Style Library

In [1]:
from pathlib import Path
from typing import List
import json

MUICE_PATH = Path("../Dataset/Muice/train.jsonl")
HARUHI_PATH = Path(r".\data\raw\Haruhi\Haruhi_clean.jsonl")
WUKONG_PATH = Path(r".\data\raw\wukong\wukong_dialog_qa.jsonl")

# Check if Dataset exist
assert MUICE_PATH.is_file(), "请确保 Muice Dataset 的路径正确！"
assert HARUHI_PATH.is_file(), "请确保 Haruhi Dataset 的路径正确！"
assert WUKONG_PATH.is_file(), "请确保 Wukong Dataset 的路径正确！"

# Load Muice Dataset
dataset_lines = MUICE_PATH.read_text(encoding="utf-8").splitlines()

muice_responses: List[str] = []

for line in dataset_lines:
    item = json.loads(line)
    muice_responses.append(item["Response"])

# Load Haruhi Dataset
dataset_lines = HARUHI_PATH.read_text(encoding="utf-8").splitlines()

ayaka_responses: List[str] = []
zhongli_responses: List[str] = []
hutao_responses: List[str] = []
haruhi_responses: List[str] = []
liyunlong_responses: List[str] = []
sheldon_responses: List[str] = []

for line in dataset_lines:
    item = json.loads(line)

    # 只提取单一角色的训练集
    if item["agent_role"] == "神里绫华":
        ayaka_responses.append(item["agent_response"])
    elif item["agent_role"] == "钟离":
        zhongli_responses.append(item["agent_response"])
    elif item["agent_role"] == "胡桃":
        hutao_responses.append(item["agent_response"])
    elif item["agent_role_name_en"] == "haruhi":
        haruhi_responses.append(item["agent_response"])
    elif item["agent_role"] == "李云龙":
        liyunlong_responses.append(item["agent_response"])
    elif item["agent_role"] == "Sheldon":
        sheldon_responses.append(item["agent_response"])

# 加载 Wukong Dataset
dataset_lines = WUKONG_PATH.read_text(encoding="utf-8").splitlines()

wukong_responses: List[str] = []
for line in dataset_lines:
    item = json.loads(line)
    wukong_responses.append(item["agent_response"])

# 输出所有训练集的长度

print(f"Muice Responses: {len(muice_responses)}")
print(f"Ayaka Responses: {len(ayaka_responses)}")
print(f"Zhongli Responses: {len(zhongli_responses)}")
print(f"Hutao Responses: {len(hutao_responses)}")
print(f"Haruhi Responses: {len(haruhi_responses)}")
print(f"Liyunlong Responses: {len(liyunlong_responses)}")
print(f"Sheldon Responses: {len(sheldon_responses)}")
print(f"Wukong Responses: {len(wukong_responses)}")

total_dataset_size = len(muice_responses) + len(ayaka_responses) + len(zhongli_responses) + len(hutao_responses) + len(haruhi_responses) + len(liyunlong_responses) + len(sheldon_responses) + len(wukong_responses)

print(f"Total dataset size: {total_dataset_size}")

Muice Responses: 3402
Ayaka Responses: 1416
Zhongli Responses: 514
Hutao Responses: 845
Haruhi Responses: 1100
Liyunlong Responses: 1782
Sheldon Responses: 1886
Wukong Responses: 700
Total dataset size: 11645


## Initialize LLM Loader

In [1]:
import os
from dataclasses import dataclass, field
from typing import Optional, Dict, Any, Sequence, Tuple, Iterable

from openai import OpenAI
from openai.types.chat import ChatCompletionMessageParam as ChatMsgParam
from dotenv import load_dotenv

load_dotenv()  # 从 .env 文件加载环境变量

# === 配置项 ===
DEFAULT_API_KEY = os.getenv("OPENAI_API_KEY", "sk-PLACEHOLDER")
DEFAULT_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://dashscope.aliyuncs.com/compatible-mode/v1")
DEFAULT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "qwen3-max-2025-09-23")
DEFAULT_SYSTEM_PROMPT = "你是一个语言风格分析专家，你需要根据通过一个带有特定角色风格的句子，改写为语义相同但风格中性的版本。要求不使用任何语气词、感叹词或修辞；不体现说话者的性格、情绪或身份；使用普通的书面语或口语表达"

ConversationTurn = Tuple[str, Optional[str]]
"""表示一次对话轮次：(user_message, assistant_reply)。assistant_reply 可为 None。"""


def _build_messages(
    prompt: str,
    history: Optional[Sequence[ConversationTurn]] = None,
    system_prompt: Optional[str] = None,
) -> list[ChatMsgParam]:
    messages: list[ChatMsgParam] = []

    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    if history:
        for user_msg, assistant_msg in history:
            messages.append({"role": "user", "content": user_msg})
            if assistant_msg:
                messages.append({"role": "assistant", "content": assistant_msg})

    messages.append({"role": "user", "content": prompt})
    return messages


@dataclass(slots=True)
class SimpleLLMClient:
    """极简 LLM 封装：初始化固定模型，提供 ask() 返回字符串。"""

    model: str = DEFAULT_MODEL
    api_key: str = DEFAULT_API_KEY
    base_url: Optional[str] = DEFAULT_BASE_URL
    system_prompt: Optional[str] = DEFAULT_SYSTEM_PROMPT
    extra_headers: Optional[Dict[str, str]] = None
    _client: OpenAI = field(init=False, repr=False)

    def __post_init__(self) -> None:
        self._client = OpenAI(
            api_key=self.api_key,
            base_url=self.base_url,
            default_headers=self.extra_headers if self.extra_headers else None,
        )

    def _consume_stream(self, stream_resp: Iterable[Any]) -> str:
        """消费流式响应，拼接内容。"""
        chunks: list[str] = []
        for chunk in stream_resp:
            choices = getattr(chunk, "choices", None)
            if not choices:
                continue
            delta = getattr(choices[0], "delta", None)
            if delta and getattr(delta, "content", None):
                chunks.append(delta.content)
        return "".join(chunks)

    def ask(
        self,
        prompt: str,
        history: Optional[Sequence[ConversationTurn]] = None,
        *,
        temperature: float = 0.7,
        top_p: float = 0.9,
        max_tokens: Optional[int] = 512,
        **kwargs: Any,
    ) -> str:
        """生成回复。

        参数：
            prompt: 当前用户输入。
            history: 可选的历史 [(user, assistant), ...]，assistant 允许为 None。
            temperature/max_tokens/stream/kwargs：直接透传给 OpenAI Chat Completion。
        返回：
            模型回复的纯文本（若响应为空则返回空字符串）。
        """
        messages = _build_messages(
            prompt=prompt,
            history=history,
            system_prompt=self.system_prompt,
        )

        response = self._client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens,
            top_p=top_p,
            stream=False,
            extra_body={"enable_thinking": False},
            **kwargs,
        )

        choice = response.choices[0]
        if hasattr(choice, "message") and getattr(choice.message, "content", None):
            return choice.message.content  # type: ignore[return-value]
        # 兼容历史版本/异常情况
        return getattr(choice, "text", "")


# 初始化一个通用实例，供 Notebook 其他单元直接调用
llm = SimpleLLMClient()

## Call LLM to Generate Neutral Sentences

In [3]:
import os
import random
from concurrent.futures import ThreadPoolExecutor, as_completed
from time import sleep
from typing import Dict, Optional, Sequence
from tqdm import tqdm

DEFAULT_TEMPERATURE = float(os.getenv("NEUTRAL_TEMPERATURE", "0.2"))
DEFAULT_SLEEP_SECONDS = float(os.getenv("NEUTRAL_SLEEP_SECONDS", "0.3"))
DEFAULT_MAX_WORKERS = int(os.getenv("NEUTRAL_MAX_WORKERS", "6"))
DEFAULT_MAX_RETRIES = int(os.getenv("NEUTRAL_MAX_RETRIES", "4"))
DEFAULT_BACKOFF_BASE_SECONDS = float(os.getenv("NEUTRAL_BACKOFF_BASE_SECONDS", "0.5"))

SHOTS_EXAMPLE = [("风格句: 我今晚吃了楠符电池呢！你要不要也来一块？（递上）\n中性句: ", "我今天吃了电池。"), ("风格句: 我才不是什么地雷女呢，只是长得像而已...\n中性句: ", "我不是地雷女。")]


def build_neutral_prompt(stylized_text: str) -> str:
    """根据带风格的原句构造改写提示。"""
    return (
        f"风格句: {stylized_text}\n"
        "中性句: "
    )


def _rewrite_with_retry(
    text: str,
    *,
    temperature: float,
    sleep_seconds: float,
    max_retries: int,
    backoff_base_seconds: float,
) -> str:
    last_err: Optional[Exception] = None
    for attempt in range(max_retries + 1):
        try:
            neutral_sentence = llm.ask(
                prompt=build_neutral_prompt(text),
                history=None,
                temperature=temperature,
            ).strip()
            if sleep_seconds > 0:
                sleep(sleep_seconds)
            return neutral_sentence
        except Exception as exc:
            last_err = exc
            if attempt >= max_retries:
                break
            jitter = random.uniform(0, 0.2)
            delay = backoff_base_seconds * (2 ** attempt) + jitter
            sleep(delay)
    if last_err is not None:
        raise last_err
    raise RuntimeError("neutral rewrite failed without explicit error")


def generate_neutral_corpus_with_CoT(
    stylized_texts: Sequence[str],
    character: Optional[str] = None,
    *,
    limit: Optional[int] = None,
    temperature: float = DEFAULT_TEMPERATURE,
    sleep_seconds: float = DEFAULT_SLEEP_SECONDS,
    max_workers: int = DEFAULT_MAX_WORKERS,
    max_retries: int = DEFAULT_MAX_RETRIES,
    backoff_base_seconds: float = DEFAULT_BACKOFF_BASE_SECONDS,
    keep_failed_as_empty: bool = True,
) -> list[Dict[str, str]]:
    """并发调用 llm.ask 批量生成中性句，内置重试与退避。"""
    source_texts = list(stylized_texts if limit is None else stylized_texts[:limit])
    total = len(source_texts)
    if total == 0:
        return []

    records_by_index: list[Optional[Dict[str, str]]] = [None] * total

    def worker(idx: int, text: str) -> tuple[int, str, Optional[Exception]]:
        try:
            neutral = _rewrite_with_retry(
                text,
                temperature=temperature,
                sleep_seconds=sleep_seconds,
                max_retries=max_retries,
                backoff_base_seconds=backoff_base_seconds,
            )
            return idx, neutral, None
        except Exception as exc:
            return idx, "", exc

    with ThreadPoolExecutor(max_workers=max(1, max_workers)) as executor:
        futures = [executor.submit(worker, idx, text) for idx, text in enumerate(source_texts)]
        for future in tqdm(
            as_completed(futures),
            total=total,
            desc=f"Generating {character or 'neutral'} neutral sentences",
        ):
            idx, neutral_sentence, err = future.result()
            original = source_texts[idx]
            if err is not None and not keep_failed_as_empty:
                records_by_index[idx] = None
                continue
            records_by_index[idx] = {"original": original, "neutral": neutral_sentence}

    # 保持输出顺序与输入一致，兼容后续 save_records。
    return [record for record in records_by_index if record is not None]

**Define the output function**

In [4]:
final_outputs = []
OUTPUT_PATH = Path("./data/neutral_sentences.jsonl")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

def save_records(character: str, records: list[Dict[str, str]]):
    for record in records:
        final_outputs.append({
            "character": character,
            "original": record["original"],
            "neutral": record["neutral"]
        })

**V1 Dataset Generate**

In [ ]:
muice_neutral_records = generate_neutral_corpus_with_CoT(muice_responses)
save_records("Muice", muice_neutral_records)
ayaka_neutral_records = generate_neutral_corpus_with_CoT(ayaka_responses)
save_records("Ayaka", ayaka_neutral_records)
zhongli_neutral_records = generate_neutral_corpus_with_CoT(zhongli_responses)
save_records("Zhongli", zhongli_neutral_records)
hutao_neutral_records = generate_neutral_corpus_with_CoT(hutao_responses)
save_records("Hutao", hutao_neutral_records)
haruhi_neutral_records = generate_neutral_corpus_with_CoT(haruhi_responses)
save_records("Haruhi", haruhi_neutral_records)

OUTPUT_PATH = Path("./data/neutral_sentences.jsonl")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_PATH.open("a", encoding="utf-8") as f:
    for item in final_outputs:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

**V2 Dataset Generate**

In [ ]:
# Clear final_outputs to ensure we don't duplicate entries when we save the next character's records.
final_outputs.clear()

liyunlong_neutral_records = generate_neutral_corpus_with_CoT(liyunlong_responses)
save_records("Liyunlong", liyunlong_neutral_records)
sheldon_neutral_records = generate_neutral_corpus_with_CoT(sheldon_responses)
save_records("Sheldon", sheldon_neutral_records)
wukong_neutral_records = generate_neutral_corpus_with_CoT(wukong_responses)
save_records("Wukong", wukong_neutral_records)

with OUTPUT_PATH.open("a", encoding="utf-8") as f:
    for item in final_outputs:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

Generating neutral neutral sentences: 100%|██████████| 1856/1856 [05:41<00:00,  5.43it/s]


## Export Neutral Sentence Results

# CoT Generation

## Load Style Library

In [2]:
from pathlib import Path
from typing_extensions import TypedDict
import json

class NeutralSentenceItem(TypedDict):
    character: str
    original: str
    neutral: str


neutral_path = Path("./data/neutral_sentences.jsonl")
neutral_items: list[NeutralSentenceItem] = []

with neutral_path.open("r", encoding="utf-8") as f:
    for line in f.readlines():
        neutral_items.append(json.loads(line))

print(f"Loaded {len(neutral_items)} neutral sentence items.")

Loaded 10965 neutral sentence items.


## Load Style Library Style Vectors

In [3]:
from collections import Counter
from typing_extensions import TypedDict

# ===== Latest style vector paths (S_total) =====
STYLE_LEXICAL_DIR = Path("./outputs/style/signature/lexical")
STYLE_PUNCTUATION_SIGNATURE_FILE = Path("./outputs/style/signature/punctuation/symbol_signature.json")
STYLE_DENSE_FILE = Path("./outputs/style/dense/normalized/style_vector_phase2_minmax.json")
STYLE_SEMANTIC_DIR = Path("./outputs/style/semantic/prag_vectors")

DENSE_KEYS = [
    "mdd",
    "verb_noun_ratio",
    "question_ratio",
    "imperative_ratio",
    "fragment_ratio",
    "marker_density",
]

CHARACTER_META = {
    "Muice": {"zh": "沐雪", "stem": "muice", "dense_role": "沐雪"},
    "Ayaka": {"zh": "神里绫华", "stem": "ayaka", "dense_role": "神里绫华"},
    "Zhongli": {"zh": "钟离", "stem": "zhongli", "dense_role": "钟离"},
    "Hutao": {"zh": "胡桃", "stem": "hutao", "dense_role": "胡桃"},
    "Haruhi": {"zh": "凉宫春日", "stem": "haruhi", "dense_role": "凉宫春日"},
    "Liyunlong": {"zh": "李云龙", "stem": "liyunlong", "dense_role": "李云龙"},
    "Sheldon": {"zh": "Sheldon", "stem": "sheldon", "dense_role": "Sheldon"},
    "Wukong": {"zh": "孙悟空", "stem": "wukong", "dense_role": "孙悟空"},
}

class RawSemanticItem(TypedDict):
    prompt: str
    response: str
    pragmatic_styles: list[dict[str, float]]

def get_lexical_keywords_from_file(file: Path, top_n: int = 25) -> list[str]:
    with open(file, "r", encoding="utf-8") as f:
        data: dict[str, float] = json.loads(f.read())
    return list(data.keys())[:top_n]

def load_punctuation_signatures(file: Path) -> dict[str, list[str]]:
    rows = json.loads(file.read_text(encoding="utf-8"))
    return {row["role"]: row.get("S_signature", []) for row in rows if row.get("role")}

def _to_float(value: object, default: float = 0.0) -> float:
    try:
        return float(value)  # type: ignore[arg-type]
    except (TypeError, ValueError):
        return default

def load_dense_metrics(file: Path) -> dict[str, dict[str, float]]:
    rows = json.loads(file.read_text(encoding="utf-8"))
    role_to_dense: dict[str, dict[str, float]] = {}

    for row in rows:
        role = row.get("role")
        if not role:
            continue
        role_to_dense[role] = {
            "mdd": _to_float(row.get("mdd", row.get("MDD", 0.0))),
            "verb_noun_ratio": _to_float(row.get("verb_noun_ratio", 0.0)),
            "question_ratio": _to_float(row.get("question_ratio", 0.0)),
            "imperative_ratio": _to_float(row.get("imperative_ratio", 0.0)),
            "fragment_ratio": _to_float(row.get("fragment_ratio", 0.0)),
            "marker_density": _to_float(row.get("marker_density", 0.0)),
        }

    return role_to_dense

def load_semantic_vectors(jsonl_file: Path, threshold: float = 0.4) -> list[tuple[str, list[str]]]:
    items: list[tuple[str, list[str]]] = []
    with open(jsonl_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            raw_item: RawSemanticItem = json.loads(line)
            score_map: dict[str, float] = {}
            for vec in raw_item.get("pragmatic_styles", []):
                score_map.update(vec)
            tags = [k for k, v in score_map.items() if float(v) > threshold]
            items.append((raw_item["response"], tags))
    return items

def build_signature_rules(lexical_keywords: list[str], punctuation_symbols: list[str]) -> list[str]:
    rules: list[str] = []
    if lexical_keywords:
        rules.append("Lexical cues: " + ", ".join(lexical_keywords[:25]))
    if punctuation_symbols:
        rules.append("Punctuation tendencies: " + ", ".join(punctuation_symbols))
    return rules

def build_style_profiles(
    top_n_lexical: int = 25,
    semantic_threshold: float = 0.4,
 ) -> tuple[dict[str, dict[str, object]], dict[tuple[str, str], list[str]]]:
    punctuation_map = load_punctuation_signatures(STYLE_PUNCTUATION_SIGNATURE_FILE)
    dense_by_role = load_dense_metrics(STYLE_DENSE_FILE)

    style_profiles: dict[str, dict[str, object]] = {}
    response_to_semantic: dict[tuple[str, str], list[str]] = {}

    for character, meta in CHARACTER_META.items():
        lexical_file = STYLE_LEXICAL_DIR / f"{meta['stem']}_pmi_filtered.json"
        lexical_keywords = get_lexical_keywords_from_file(lexical_file, top_n=top_n_lexical) if lexical_file.exists() else []

        punctuation_symbols = punctuation_map.get(meta["zh"], [])
        signature_rules = build_signature_rules(lexical_keywords, punctuation_symbols)

        dense_metrics = dense_by_role.get(meta["dense_role"], {k: 0.0 for k in DENSE_KEYS})

        semantic_file = STYLE_SEMANTIC_DIR / f"{meta['stem']}.jsonl"
        semantic_items = load_semantic_vectors(semantic_file, threshold=semantic_threshold) if semantic_file.exists() else []

        tag_counter = Counter()
        for response, tags in semantic_items:
            response_to_semantic[(character, response)] = tags
            tag_counter.update(tags)

        semantic_tags = [tag for tag, _ in tag_counter.most_common(8)]

        style_profiles[character] = {
            "semantic_tags": semantic_tags,
            "structural_metrics": dense_metrics,
            "signature_rules": signature_rules,
        }

    return style_profiles, response_to_semantic

STYLE_PROFILE_BY_CHARACTER, RESPONSE_TO_SEMANTIC = build_style_profiles()

# 修复部分角色的风格向量
STYLE_PROFILE_BY_CHARACTER["Muice"]["signature_rules"][1] = "Punctuation tendencies: ..., ⭐, ?!, multi-!, ♪, ❤"  # type: ignore[index]
STYLE_PROFILE_BY_CHARACTER["Wukong"]["semantic_tags"] = ["energetic", "playful", "irritable", "loyal", "confident"]

print(f"已加载 {len(STYLE_PROFILE_BY_CHARACTER)} 个角色的 S_total 风格画像")
for character, profile in STYLE_PROFILE_BY_CHARACTER.items():
    semantic_count = len(profile.get("semantic_tags", []))  # type: ignore[arg-type]
    signature_count = len(profile.get("signature_rules", []))  # type: ignore[arg-type]
    print(f"  - {character}: semantic_tags={semantic_count}, signature_rules={profile.get('signature_rules', [])}")

已加载 8 个角色的 S_total 风格画像
  - Muice: semantic_tags=8, signature_rules=['Lexical cues: 喵, 沐沐, AI, ~, 沐雪, 恼, 女孩子, ⭐, 聊天, 唔, 呀, 可爱, 答, 不行, 笨蛋, 雪雪, 直播, 脸红, 睡觉, www, 谢谢, 变态, 骂, 嘛, 杂鱼', 'Punctuation tendencies: ..., ⭐, ?!, multi-!, ♪, ❤']
  - Ayaka: semantic_tags=8, signature_rules=['Lexical cues: 稻妻国, 神里家, 稻妻, 大小姐, 家族, 传统, 神, 奉行, 眼狩令, 当地, 美丽, 社, 舞蹈, 茶道, 神社, 文化, 剑术, 眼, 祭典, 旅行者, 美食, 继承, 历史, 神明, 派蒙', 'Punctuation tendencies: ⋯, ...']
  - Zhongli: semantic_tags=8, signature_rules=['Lexical cues: 岩石, 岩, 璃月, 璃, 力, 契约, 炼金术, 月, 盐, 帝君, 魔神, 操控, 岩王, 王, 封印, 元素, 大陆, 七星, 客卿, 岩元素, 大地, 学问, 力量, 凝光, 易事', 'Punctuation tendencies: ...']
  - Hutao: semantic_tags=8, signature_rules=['Lexical cues: 往生堂, 嘿嘿, 嘻嘻, 堂主, 宝藏, 可是, 哎呀呀, 哦哦哦, 诗歌, 神秘, 惊喜, 谜题, 胡桃, 有趣, 灵魂, 哈哈哈, 秘密, 哇, 奇妙, 意想不到, 隐藏, 亡灵, 生死, 冒险, 巫师', 'Punctuation tendencies: ..., ♪']
  - Haruhi: semantic_tags=8, signature_rules=['Lexical cues: SOS, 阿虚, 社团, 团, 哼, 学校, 事件, 朝比奈, 超自然, 文化祭, 吸引, 古泉, 创意, 实玖瑠, 与众不同, 创新, 电影, 束缚, 凉宫, 主题, 外星人, 学姐, 举办, 奇怪, 加入', 'Punctuation t

## Call LLM to Generate CoT Process

In [4]:
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from time import sleep
from typing import Dict
from tqdm import tqdm
import re

DEFAULT_TEMPERATURE = float(os.getenv("NEUTRAL_TEMPERATURE", "0.4"))
DEFAULT_TOP_P = float(os.getenv("NEUTRAL_TOP_P", "0.9"))
DEFAULT_SLEEP_SECONDS = float(os.getenv("NEUTRAL_SLEEP_SECONDS", "0.3"))
MODEL = os.getenv("OPENAI_CHAT_MODEL", "qwen3-30b-a3b-instruct-2507")

SYSTEM_PROMPT = """你是一个角色风格分析器。
给定中性句、风格句和三层风格向量 S_total=<v_dense, t_semantic, z_signature>，请输出简短思维链。
要求：
1) 解释角色如何从语义标签（t_semantic）选择语气；
2) 解释如何依据 6 维结构向量（v_dense）调整句式节奏；
3) 解释如何应用身份签名（z_signature）中的词汇和标点偏好。
思维链输出内容最多100字，着重于推理风格向量插入到中性句中的过程。
"""

SHOTS_EXAMPLE = [
    (
        "角色: Katsumi\n"
        "中性句: 谢谢你帮我。\n"
        "风格句: 哼！才、才没想帮你呢，笨蛋！\n"
        "Semantic Tags: tsundere, defensive\n"
        "Dense Metrics(6D): mdd=0.45, verb_noun_ratio=0.55, question_ratio=0.18, imperative_ratio=0.26, fragment_ratio=0.48, marker_density=0.29\n"
        "Signature Rules: Lexical cues: 哼, 才没有, 笨蛋; Punctuation tendencies: ?!\n",
        "她先用傲娇和防御语义决定否认感谢，再用中高碎片率和祈使感压缩句式，最后用“哼”“笨蛋”与感叹/反问标点完成身份指纹。"
    ),
    (
        "角色: Lady Elara\n"
        "中性句: 我不知道这是否是个好主意。\n"
        "风格句: 或许……我们还需要再仔细考虑，这是否真是最稳妥的选择。\n"
        "Semantic Tags: serious, rational, elegant\n"
        "Dense Metrics(6D): mdd=0.62, verb_noun_ratio=0.31, question_ratio=0.08, imperative_ratio=0.12, fragment_ratio=0.15, marker_density=0.10\n"
        "Signature Rules: Lexical cues: 考量, 契约, 公平; Punctuation tendencies: ...\n",
        "她以理性和庄重语义选择审慎措辞，较高mdd与低碎片率使句子更完整克制，再用“或许”“稳妥”及省略号形成优雅且保留余地的表达。"
    ),
]

EN_NAME_TO_ZH = {
    "Muice": "沐雪",
    "Ayaka": "神里绫华",
    "Zhongli": "钟离",
    "Hutao": "胡桃",
    "Haruhi": "凉宫春日",
    "Liyunlong": "李云龙",
    "Sheldon": "Sheldon",
    "Wukong": "孙悟空",
}

llm = SimpleLLMClient(MODEL, DEFAULT_API_KEY, system_prompt=SYSTEM_PROMPT)

def _format_dense_metrics(dense_metrics: dict[str, float]) -> str:
    ordered = [f"{k}={dense_metrics.get(k, 0.0):.4f}" for k in DENSE_KEYS]
    return ", ".join(ordered)

def _safe_profile(character_en: str) -> dict[str, object]:
    raw = STYLE_PROFILE_BY_CHARACTER.get(character_en, {})
    return raw if isinstance(raw, dict) else {}

def _safe_structural_metrics(profile: dict[str, object]) -> dict[str, float]:
    raw = profile.get("structural_metrics")
    result = {k: 0.0 for k in DENSE_KEYS}
    if isinstance(raw, dict):
        for key in DENSE_KEYS:
            result[key] = _to_float(raw.get(key, 0.0))
    return result

def _safe_signature_rules(profile: dict[str, object]) -> list[str]:
    raw = profile.get("signature_rules")
    if not isinstance(raw, list):
        return []
    return [str(x) for x in raw]

def _safe_semantic_tags(profile: dict[str, object]) -> list[str]:
    raw = profile.get("semantic_tags")
    if not isinstance(raw, list):
        return []
    return [str(x) for x in raw]

def build_cot_prompt(
    character: str,
    neutral_sentence: str,
    stylized_text: str,
    semantic_tags: list[str],
    structural_metrics: dict[str, float],
    signature_rules: list[str],
) -> str:
    semantic_text = ", ".join(semantic_tags) if semantic_tags else "None"
    signature_text = "; ".join(signature_rules) if signature_rules else "None"
    dense_text = _format_dense_metrics(structural_metrics)
    return (
        f"角色: {character}\n"
        f"中性句: {neutral_sentence}\n"
        f"风格句: {stylized_text}\n"
        f"Semantic Tags: {semantic_text}\n"
        f"Dense Metrics(6D): {dense_text}\n"
        f"Signature Rules: {signature_text}\n"
    )

def find_sentence_semantic_tags(character_en: str, stylized_text: str) -> list[str]:
    tags = RESPONSE_TO_SEMANTIC.get((character_en, stylized_text))
    if tags is not None and len(tags) > 0:
        return tags
    profile = _safe_profile(character_en)
    return _safe_semantic_tags(profile)

_THINK_RE = re.compile(r"<\s*think\s*>(.*?)<\s*/\s*think\s*>", re.IGNORECASE | re.DOTALL)

def _extract_think_block(text: str) -> str | None:
    if not text:
        return None
    text = text.strip()

    if text.startswith("<think>") and text.endswith("</think>"):
        return text

    m = _THINK_RE.search(text)
    if m:
        return f"<think>{m.group(1).strip()}</think>"

    s2 = re.sub(r"^```(?:[a-zA-Z0-9_-]+)?\s*|\s*```$", "", text, flags=re.DOTALL).strip()
    m = _THINK_RE.search(s2)
    if m:
        return f"<think>{m.group(1).strip()}</think>"
    return f"<think>{text}</think>" if text else None

def generate_neutral_corpus_with_CoT(
    neutral_sentences: list[NeutralSentenceItem],
    *,
    temperature: float = DEFAULT_TEMPERATURE,
    top_p: float = DEFAULT_TOP_P,
    sleep_seconds: float = DEFAULT_SLEEP_SECONDS,
    max_workers: int = 6,
 ) -> list[Dict[str, str]]:
    """并发调用 llm.ask 批量生成 CoT，并确保输出为 <think>...</think> 格式。"""
    records_by_index: list[Dict[str, str] | None] = [None] * len(neutral_sentences)
    MAX_RETRIES = 2

    def worker(index: int, item: NeutralSentenceItem) -> tuple[int, Dict[str, str] | None]:
        character_en = item["character"]
        character_zh = EN_NAME_TO_ZH.get(character_en, character_en)

        profile = _safe_profile(character_en)
        structural_metrics = _safe_structural_metrics(profile)
        signature_rules = _safe_signature_rules(profile)
        semantic_tags = find_sentence_semantic_tags(character_en, item["original"])

        prompt = build_cot_prompt(
            character_zh,
            item["neutral"],
            item["original"],
            semantic_tags,
            structural_metrics,
            signature_rules,
        )

        thinking_process: str | None = None
        for _ in range(MAX_RETRIES + 1):
            result = llm.ask(
                prompt=prompt,
                history=SHOTS_EXAMPLE,
                temperature=temperature,
                top_p=top_p,
            ).strip()
            thinking_process = _extract_think_block(result)
            if thinking_process:
                break

        if thinking_process is None:
            return index, None

        return index, {
            "character": item["character"],
            "original": item["original"],
            "neutral": item["neutral"],
            "CoT": thinking_process,
        }

    with ThreadPoolExecutor(max_workers=max(1, max_workers)) as executor:
        futures = [executor.submit(worker, index, item) for index, item in enumerate(neutral_sentences)]
        for future in tqdm(as_completed(futures), total=len(futures), desc="Generating CoT"):
            index, record = future.result()
            records_by_index[index] = record
            if sleep_seconds > 0:
                sleep(sleep_seconds)

    records = [record for record in records_by_index if record is not None]
    failed = len(neutral_sentences) - len(records)
    if failed > 0:
        print(f"本批 CoT 生成失败: {failed} 条")
    return records

In [6]:
# 分批生成并可恢复的 CoT 生成与保存

OUTPUT_PATH = Path("./data/neutral_sentences_with_CoT.jsonl")
BATCH_SIZE = 100             # 每批处理条数

def _load_done_keys(path: Path) -> set[tuple[str, str]]:
    """从已存在的 JSONL 中读取已完成项的键 (character, original)。"""
    done: set[tuple[str, str]] = set()
    if path.is_file():
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                    done.add((obj["character"], obj["original"]))
                except Exception:
                    # 跳过坏行
                    continue
    return done


def _append_jsonl(path: Path, items: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        for item in items:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")


# 读取进度
done_keys = _load_done_keys(OUTPUT_PATH)
total = len(neutral_items)
remaining_items = [it for it in neutral_items if (it["character"], it["original"]) not in done_keys]

print(f"Total: {total}, Done: {len(done_keys)}, Remaining: {len(remaining_items)}")

if not remaining_items:
    print("All items are already processed.")
else:
    start_done = len(done_keys)
    batches_run = 0

    # 逐批处理
    while remaining_items:
        batch = remaining_items[:BATCH_SIZE]
        remaining_items = remaining_items[BATCH_SIZE:]

        print(f"Processing batch {batches_run + 1}: {len(batch)} items...")
        records = generate_neutral_corpus_with_CoT(batch)

        # 追加保存，便于中断后恢复
        _append_jsonl(OUTPUT_PATH, records)

        # 更新已完成键，便于重复运行时即时跳过
        for r in records:
            done_keys.add((r["character"], r["original"]))

        print(f"Saved {len(records)} items. Progress: {len(done_keys)}/{total}")
        batches_run += 1

    print(f"Run finished. New processed: {len(done_keys) - start_done}. Remaining: {total - len(done_keys)}")

Total: 10965, Done: 10836, Remaining: 0
All items are already processed.
